In [ ]:
import torch
torch.__version__

In [ ]:
import math
import torch.nn as nn

In [ ]:
class MoE_Router(nn.Module):
    def __init__(self, embed_dim, num_experts, counts):
        super().__init__()
        self.num_experts = num_experts
        self.counts = counts

        self.weights = nn.Parameter(torch.randn(embed_dim, num_experts)) #[embed_dim, num_experts]

        # inputs  [batch_size, seq_len, embed_dim]
        # routed_tokens [num_experts * counts, embed_dim]
        # 1st 'counts' slots belong to Expert_0, next 'counts' slots to Expert_1, so on.
        # counts - No. of tokens each expert can handle (capacity)

    def forward(self, inputs): # Taking inputs
        # compute top-k rounting
        batch_size, seq_len, embed_dim = inputs.shape
        flat_inputs = inputs.view(-1, embed_dim) # flatten to [B*S, E]
        num_tokens = flat_inputs.shape[0]

        # [B*S, E] @ [E, N] -> [B*S, N]
        logits = torch.matmul(flat_inputs, self.weights)

        # Apply softmax to logits across the EXPERT dimension
        scores = torch.softmax(logits, dim=-1) # [B*S, N]

        # Get Top-1 expert choices per token
        # top_scores: [B*S], expert_ids: [B*S]
        top_scores, expert_ids = torch.topk(scores, k=1, dim=-1)
        top_scores = top_scores.squeeze(-1) # Remove dims with size = 1
        expert_ids = expert_ids.squeeze(-1) # [B*S, 1] -> [B*S]

        # Route tokens using Boolean Masking while respecting capacity
        routed_tokens = torch.zeros(self.num_experts * self.counts, embed_dim,
                                    device=inputs.device)
        # Keep track of which original token index ended up in which slot
        # -1 indicates an empty/unassigned slot
        slot_to_token_idx = torch.full((self.num_experts * self.counts,), -1,
                                       dtype=torch.long, device=inputs.device)

        for expert_id in range(num_experts):
            # boolean mask for tokens selecting this specfic expert
            expert_mask = (expert_ids == expert_id) #[T, F, F, T, ...] #[B*S]

            # select all tokens which prefer this expert
            selected_tokens = flat_inputs[expert_mask]

            # get the global token positions  that selected this expert
            global_indices = torch.nonzero(expert_mask).squeeze(-1) #[B*S]

            # Cap the tokens if they exceed capacity (counts)
            actual_count = min(selected_tokens.shape[0], counts)

            if actual_count > 0:
               # Calculate the exact destination slice in our output slot tensor
               start_slot = expert_id * self.counts
               end_slot = start_slot + actual_count

               routed_tokens[start_slot:end_slot] = selected_tokens[:actual_count]
               slot_to_token_idx[start_slot:end_slot] = global_indices[:actual_count]

        return routed_tokens, slot_to_token_idx

In [ ]:
batch_size = 4
seq_len = 100
embed_dim = 512

num_experts = 64
counts = 8

router = MoE_Router(embed_dim, num_experts, counts)
X = torch.randn(batch_size, seq_len, embed_dim)

routed_outputs, slot_mapping = router.forward(X)
print("Routed tokens shape:", routed_outputs.shape) # Expected: [3 * 3, 16] -> [9, 16]
print("Slot assignment map:", slot_mapping)        # Maps slot index to original flat token index